<a href="https://colab.research.google.com/github/HuriyaAhmerr/5782076-AnalyticsEngineering/blob/main/5782076_library_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# Community library data: create your own dataset (individual assignment)

You will work with data from a fictional community library service, covering the calendar year 2025. **Each student gets their own version.** The data are generated from your student ID, so:

- the same ID always gives the same data, so you can come back and re-create it;
- different IDs give different rows, values and problems;
- the layout of one table can differ from a classmate's, so check your data dictionary rather than relying on someone else's column names.

The files are exactly as exported from the library's systems. Checking them before you rely on them is part of the task.

Run the cells below in order.

## Step 1: Add the generator

This cell saves the data generator to the notebook's files. You do not need to edit it. Run it once.

In [ ]:
%%writefile generate_library_data_by_id.py
#!/usr/bin/env python3
"""
Community-library dataset, personalised by student ID (fictional data).

Student use (Colab or terminal):
    from generate_library_data_by_id import generate
    generate(input("Enter your student ID: "))
  or:  python generate_library_data_by_id.py          (prompts for the ID)
       python generate_library_data_by_id.py 1234567

What the ID does
  1. SEED: the ID is hashed into a random seed, so the same ID always gives the
     same dataset, and different IDs give different rows, values and mess.
  2. LAYOUT VARIANT: the last digit of the ID selects one of five layouts.
     The underlying data and the injected problems are the same in kind and
     amount for everyone; the variant changes how one table is laid out.
        last digit 1, 2 -> A  baseline layout
        last digit 3, 4 -> B  fines stored as whole pence (fine_pence)
        last digit 5, 6 -> C  loans split over two files (first/second half of year)
        last digit 7, 8 -> D  loans give branch NAME (branch), not branch_id
        last digit 9, 0 -> E  item type and category merged (classification)

Marker use
    python generate_library_data_by_id.py 1234567 --marker
    python generate_library_data_by_id.py --ids ids.txt --marker   (one ID per line)
  writes marker_lists/marker_<ID>.md (issue counts, reference numbers) and, for
  --ids, marker_lists/variant_summary.csv.
  NOTE: students can read this script. It does not write the marker list unless
  asked, but the issue types are visible in the code. See the staff notes.
"""
import argparse
import hashlib
import os
import re
import sys
import tempfile
from datetime import date, timedelta

import numpy as np
import pandas as pd

VERSION = "library-v1"
START, END = date(2025, 1, 1), date(2025, 12, 31)
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d %b %Y"]
DATE_FORMAT_PROBS = [0.78, 0.17, 0.05]
VARIANT_BY_DIGIT = {"1": "A", "2": "A", "3": "B", "4": "B", "5": "C", "6": "C",
                    "7": "D", "8": "D", "9": "E", "0": "E"}
VARIANT_TEXT = {
    "A": "Baseline layout.",
    "B": "Fines stored as whole pence in loans.fine_pence (integers), not pounds.",
    "C": "Loans split over loans_2025_H1.csv (Jan-Jun) and loans_2025_H2.csv (Jul-Dec); "
         "H2 has its columns in a different order.",
    "D": "loans has the branch NAME in column 'branch' instead of branch_id; join to branches on branch_name.",
    "E": "items has one 'classification' column (\"Type / Category\") instead of item_type and category.",
}


# ------------------------------------------------------------------ helpers
def seed_from_id(student_id):
    digest = hashlib.sha256(f"{VERSION}:{student_id.strip().upper()}".encode()).digest()
    return int.from_bytes(digest[:8], "big")


def variant_from_id(student_id):
    digits = re.findall(r"\d", student_id)
    if not digits:
        raise ValueError("Student ID must contain at least one digit.")
    return VARIANT_BY_DIGIT[digits[-1]]


def clean_id(student_id):
    sid = str(student_id).strip()
    if not sid:
        raise ValueError("Student ID is empty.")
    return sid


def fmt_date(d, f):
    return "" if d is None or pd.isna(d) else d.strftime(f)


def parse_date(s):
    s = str(s).strip()
    if not s or s == "nan":
        return pd.NaT
    if "/" in s:
        return pd.to_datetime(s, format="%d/%m/%Y")
    if any(ch.isalpha() for ch in s):
        return pd.to_datetime(s, format="%d %b %Y")
    return pd.to_datetime(s, format="%Y-%m-%d")


# ------------------------------------------------------- baseline generation
def build_baseline(seed, out):
    """Writes baseline-layout CSVs into `out`; returns the injected-issue log."""
    rng = np.random.default_rng(seed)
    os.makedirs(out, exist_ok=True)
    log = {}

    def pick_idx(n, k):
        return rng.choice(n, size=k, replace=False)

    # ---- branches
    branch_weights = np.array([0.22, 0.12, 0.15, 0.08, 0.14, 0.10, 0.11, 0.08])
    branches = pd.DataFrame({
        "branch_id": [f"B{i:02d}" for i in range(1, 9)],
        "branch_name": ["Abbeygate", "Brookfield", "Castlemere", "Dunmore",
                        "Elmstead", "Fairholme", "Greystone", "Harlow Green"],
        "area": ["North", "North", "East", "East", "South", "South", "West", "West"],
        "opened_year": [1962, 1974, 1988, 1995, 1971, 2003, 1958, 2011],
    })

    # ---- items
    N_ITEMS = 1200
    adjectives = ["Quiet", "Last", "Hidden", "Silver", "Northern", "Broken", "Golden",
                  "Winter", "Distant", "Little", "Secret", "Burning", "Paper", "Wild"]
    nouns = ["River", "Garden", "Letters", "Harbour", "Orchard", "Kingdom", "Season",
             "Map", "Bridge", "Lighthouse", "Stranger", "Engine", "Forest", "Promise"]
    item_types = rng.choice(["Book", "DVD", "Audiobook"], N_ITEMS, p=[0.75, 0.15, 0.10])
    category_by_type = {
        "Book": (["Fiction", "Non-fiction", "Children", "Reference"], [0.40, 0.30, 0.25, 0.05]),
        "DVD": (["Film", "Children"], [0.65, 0.35]),
        "Audiobook": (["Fiction", "Non-fiction"], [0.60, 0.40]),
    }
    cost_range = {"Book": (6, 25), "DVD": (8, 20), "Audiobook": (12, 30)}
    categories, costs = [], []
    for t in item_types:
        cats, probs = category_by_type[t]
        categories.append(rng.choice(cats, p=probs))
        lo, hi = cost_range[t]
        costs.append(round(float(rng.uniform(lo, hi)), 2))
    items = pd.DataFrame({
        "item_id": [f"I{i:04d}" for i in range(1, N_ITEMS + 1)],
        "title": [f"The {rng.choice(adjectives)} {rng.choice(nouns)}" for _ in range(N_ITEMS)],
        "item_type": item_types,
        "category": categories,
        "replacement_cost": costs,
        "home_branch_id": rng.choice(branches["branch_id"], N_ITEMS, p=branch_weights),
    })
    clean_items = items.copy()

    # ---- members
    N_MEMBERS = 800
    age_bands = ["Under 12", "12-17", "18-34", "35-59", "60+"]
    bands = rng.choice(age_bands, N_MEMBERS, p=[0.14, 0.12, 0.22, 0.30, 0.22])

    def membership(b):
        if b in ("Under 12", "12-17"):
            return "Junior"
        if b == "60+" and rng.random() < 0.6:
            return "Concession"
        return "Standard"

    join_days = rng.integers(0, (date(2025, 10, 31) - date(2015, 1, 1)).days, N_MEMBERS)
    letters = np.array(list("ABDEFGHJLNPQRSTUWXYZ"))
    members = pd.DataFrame({
        "member_id": [f"M{i:05d}" for i in range(1, N_MEMBERS + 1)],
        "join_date": [date(2015, 1, 1) + timedelta(int(d)) for d in join_days],
        "age_band": bands,
        "membership_type": [membership(b) for b in bands],
        "home_branch_id": rng.choice(branches["branch_id"], N_MEMBERS, p=branch_weights),
        "postcode": [f"ZZ{rng.integers(1, 10)} {rng.integers(1, 10)}{''.join(rng.choice(letters, 2))}"
                     for _ in range(N_MEMBERS)],
    })
    clean_members = members.copy()
    activity = rng.gamma(1.2, 1.0, N_MEMBERS)
    activity = activity / activity.sum()

    # ---- loans
    N_LOANS = 6000
    cat_prefs = {
        "Under 12": {"Children": .70, "Film": .10, "Fiction": .10, "Non-fiction": .05, "Reference": .05},
        "12-17": {"Fiction": .40, "Children": .20, "Film": .20, "Non-fiction": .15, "Reference": .05},
        "18-34": {"Fiction": .35, "Film": .25, "Non-fiction": .30, "Children": .02, "Reference": .08},
        "35-59": {"Fiction": .30, "Non-fiction": .35, "Film": .15, "Children": .12, "Reference": .08},
        "60+": {"Fiction": .40, "Non-fiction": .35, "Film": .12, "Children": .05, "Reference": .08},
    }
    items_by_cat = {}
    for c, grp in clean_items.groupby("category"):
        w = rng.pareto(2.0, len(grp)) + 1
        items_by_cat[c] = (grp.reset_index(drop=True), w / w.sum())

    days = [START + timedelta(d) for d in range((END - START).days + 1)]
    day_w = np.array([(2.0 if d.weekday() == 5 else 0.6 if d.weekday() == 6 else 1.0)
                      * (1.4 if d.month in (7, 8) else 0.8 if d.month == 12 else 1.0)
                      for d in days])
    loan_dates = sorted(rng.choice(len(days), N_LOANS, p=day_w / day_w.sum()))
    period = {"Book": 21, "DVD": 7, "Audiobook": 14}
    fine_rate = {"Book": 0.20, "Audiobook": 0.20, "DVD": 1.00}

    rows = []
    for n, di in enumerate(loan_dates, start=1):
        ld = days[di]
        m = clean_members.iloc[rng.choice(N_MEMBERS, p=activity)]
        prefs = cat_prefs[m["age_band"]]
        cat = rng.choice(list(prefs), p=list(prefs.values()))
        grp, w = items_by_cat[cat]
        it = grp.iloc[rng.choice(len(grp), p=w)]
        branch = m["home_branch_id"] if rng.random() < 0.75 else rng.choice(branches["branch_id"], p=branch_weights)
        p = period[it["item_type"]]
        due = ld + timedelta(p)
        still_out = ld > END - timedelta(21) and rng.random() < 0.6
        lost = (not still_out) and rng.random() < 0.015
        ret, fine = None, np.nan
        if not still_out and not lost:
            if rng.random() < 0.84:
                days_out = int(rng.integers(1, p + 1))
            else:
                days_out = p + min(int(rng.geometric(0.15)), 60)
            ret = ld + timedelta(days_out)
            late = max(0, days_out - p)
            fine = 0.0 if it["category"] == "Children" else min(10.0, round(late * fine_rate[it["item_type"]], 2))
        rows.append([f"L{n:06d}", m["member_id"], it["item_id"], branch, ld, due, ret, fine])
    loans = pd.DataFrame(rows, columns=["loan_id", "member_id", "item_id", "branch_id",
                                        "loan_date", "due_date", "return_date", "fine_charged"])

    # ================================================================ inject mess
    # ---- items
    n = len(items)
    variants = {"Fiction": ["fiction", "FICTION", "Fiction "],
                "Non-fiction": ["Non fiction", "nonfiction", "Non-Fiction", "non-fiction "],
                "Children": ["Childrens", "Children's", "children"],
                "Reference": ["reference", "Ref."], "Film": ["film", "Films"]}
    idx = pick_idx(n, int(0.12 * n))
    items["category"] = items["category"].astype(object)
    for i in idx:
        items.at[i, "category"] = rng.choice(variants[items.at[i, "category"]])
    log["items_category_variants"] = len(idx)
    items["replacement_cost"] = items["replacement_cost"].astype(object)
    idx = pick_idx(n, int(0.06 * n))
    for i in idx:
        items.at[i, "replacement_cost"] = f"£{items.at[i, 'replacement_cost']:.2f}"
    log["items_cost_as_text_with_pound_sign"] = len(idx)
    rest = np.setdiff1d(np.arange(n), idx)
    idx2 = rng.choice(rest, int(0.02 * n), replace=False)
    items.loc[idx2, "replacement_cost"] = np.nan
    log["items_cost_missing"] = len(idx2)

    # ---- members
    n = len(members)
    band_variants = {"Under 12": ["under 12", "<12"], "12-17": ["12 - 17", "12 to 17"],
                     "18-34": ["18 - 34", "18 to 34"], "35-59": ["35 - 59", "35 to 59"],
                     "60+": ["60 plus", "60+ "]}
    idx = pick_idx(n, int(0.08 * n))
    for i in idx:
        members.at[i, "age_band"] = rng.choice(band_variants[members.at[i, "age_band"]])
    log["members_age_band_variants"] = len(idx)

    jd_fmt = rng.choice(len(DATE_FORMATS), n, p=DATE_FORMAT_PROBS)
    members["join_date"] = [fmt_date(d, DATE_FORMATS[f]) for d, f in zip(members["join_date"], jd_fmt)]
    log["members_join_date_not_iso"] = int((jd_fmt != 0).sum())

    pc = members["postcode"].tolist()
    low = pick_idx(n, int(0.15 * n))
    for i in low:
        pc[i] = pc[i].lower()
    nospace = pick_idx(n, int(0.10 * n))
    for i in nospace:
        pc[i] = pc[i].replace(" ", "")
    miss = pick_idx(n, int(0.04 * n))
    for i in miss:
        pc[i] = np.nan
    members["postcode"] = pc
    log["members_postcode_lowercase"] = len(low)
    log["members_postcode_no_space"] = len(nospace)
    log["members_postcode_missing"] = len(miss)

    exact = members.iloc[pick_idx(n, int(0.03 * n))]
    cand = members[(jd_fmt == 0) & members["postcode"].notna()
                   & (members["postcode"].astype(str).str.upper() == members["postcode"].astype(str))]
    near = cand.sample(8, random_state=int(rng.integers(0, 2**31 - 1))).copy()
    near["join_date"] = [pd.to_datetime(d).strftime("%d/%m/%Y") for d in near["join_date"]]
    near["postcode"] = near["postcode"].astype(str).str.lower()
    members = pd.concat([members, exact, near]).sort_values("member_id", kind="mergesort").reset_index(drop=True)
    log["members_exact_duplicate_rows"] = len(exact)
    log["members_near_duplicate_rows"] = len(near)

    # ---- loans
    n = len(loans)
    returned = loans.index[loans["return_date"].notna()].to_numpy()
    bad_ret = rng.choice(returned, 12, replace=False)
    for i in bad_ret:
        loans.at[i, "return_date"] = loans.at[i, "loan_date"] - timedelta(int(rng.integers(1, 11)))
    log["loans_return_before_loan"] = len(bad_ret)

    pos_fine = loans.index[(loans["fine_charged"] > 0) & ~loans.index.isin(bad_ret)].to_numpy()
    pick = rng.choice(pos_fine, 9, replace=False)
    big, neg = pick[:3], pick[3:]
    loans.loc[big, "fine_charged"] = loans.loc[big, "fine_charged"] * 100
    loans.loc[neg, "fine_charged"] = -loans.loc[neg, "fine_charged"]
    log["loans_fine_100x_outliers"] = len(big)
    log["loans_fine_negative"] = len(neg)

    d_fmt = rng.choice(len(DATE_FORMATS), n, p=DATE_FORMAT_PROBS)
    for col in ["loan_date", "due_date", "return_date"]:
        loans[col] = [fmt_date(d, DATE_FORMATS[f]) for d, f in zip(loans[col], d_fmt)]
    log["loans_rows_with_non_iso_dates"] = int((d_fmt != 0).sum())

    fine_txt = []
    n_pound = n_blank_zero = 0
    for v in loans["fine_charged"]:
        if pd.isna(v):
            fine_txt.append("")
        elif v == 0:
            if rng.random() < 0.10:
                fine_txt.append(""); n_blank_zero += 1
            else:
                fine_txt.append("0.00")
        elif v > 0 and rng.random() < 0.07:
            fine_txt.append(f"£{v:.2f}"); n_pound += 1
        else:
            fine_txt.append(f"{v:.2f}")
    loans["fine_charged"] = fine_txt
    log["loans_fine_with_pound_sign"] = n_pound
    log["loans_zero_fine_left_blank"] = n_blank_zero

    bid = loans["branch_id"].tolist()
    low = pick_idx(n, int(0.02 * n))
    for i in low:
        bid[i] = bid[i].lower()
    rest = np.setdiff1d(np.arange(n), low)
    spc = rng.choice(rest, int(0.01 * n), replace=False)
    for i in spc:
        bid[i] = " " + bid[i]
    loans["branch_id"] = bid
    log["loans_branch_id_lowercase"] = len(low)
    log["loans_branch_id_leading_space"] = len(spc)

    o_mem = pick_idx(n, int(0.01 * n))
    loans.loc[o_mem, "member_id"] = [f"M{rng.integers(901, 999):05d}" for _ in o_mem]
    rest = np.setdiff1d(np.arange(n), o_mem)
    o_item = rng.choice(rest, int(0.005 * n), replace=False)
    loans.loc[o_item, "item_id"] = [f"I{rng.integers(1201, 1300):04d}" for _ in o_item]
    log["loans_member_id_not_in_members"] = len(o_mem)
    log["loans_item_id_not_in_items"] = len(o_item)

    dups = loans.iloc[pick_idx(n, int(0.01 * n))]
    loans = pd.concat([loans, dups]).sort_values("loan_id", kind="mergesort").reset_index(drop=True)
    log["loans_exact_duplicate_rows"] = len(dups)

    branches.to_csv(f"{out}/branches.csv", index=False)
    members.to_csv(f"{out}/members.csv", index=False)
    items.to_csv(f"{out}/items.csv", index=False)
    loans.to_csv(f"{out}/loans.csv", index=False)
    return log


# ---------------------------------------------------- reference numbers (marker)
def reference_numbers(base):
    raw = pd.read_csv(f"{base}/loans.csv", dtype=str, keep_default_na=False)
    c = raw.drop_duplicates().copy()
    c["branch_id"] = c["branch_id"].str.strip().str.upper()
    for col in ["loan_date", "due_date", "return_date"]:
        c[col] = c[col].map(parse_date)
    mem_ids = set(pd.read_csv(f"{base}/members.csv", dtype=str)["member_id"])
    item_ids = set(pd.read_csv(f"{base}/items.csv", dtype=str)["item_id"])
    ref = {
        "raw_loan_rows": len(raw),
        "loan_rows_after_dropping_exact_duplicates": len(c),
        "loans_with_member_not_in_members": int((~c["member_id"].isin(mem_ids)).sum()),
        "loans_with_item_not_in_items": int((~c["item_id"].isin(item_ids)).sum()),
    }
    ok = c[c["return_date"].notna() & (c["return_date"] >= c["loan_date"])]
    ref["late_return_share_of_valid_returns"] = round(float((ok["return_date"] > ok["due_date"]).mean()), 3)
    by_branch = c["branch_id"].value_counts().sort_index()
    by_month = c["loan_date"].dt.to_period("M").value_counts().sort_index()
    return ref, by_branch, by_month


# ------------------------------------------------------------ layout variants
def apply_variant(variant, base, out):
    """Reads baseline CSVs from `base`, writes the student's layout to `out`.
    Returns (files written, log keys that no longer apply, schema notes)."""
    os.makedirs(out, exist_ok=True)
    rd = lambda f: pd.read_csv(f"{base}/{f}.csv", dtype=str, keep_default_na=False)
    branches, members, items, loans = rd("branches"), rd("members"), rd("items"), rd("loans")
    dropped = []

    if variant == "B":
        def to_pence(s):
            s = s.strip().replace("£", "")
            return "" if s == "" else str(int(round(float(s) * 100)))
        loans["fine_charged"] = loans["fine_charged"].map(to_pence)
        loans = loans.rename(columns={"fine_charged": "fine_pence"})
        dropped = ["loans_fine_with_pound_sign"]
    elif variant == "D":
        name = dict(zip(branches["branch_id"], branches["branch_name"]))

        def to_name(v):
            core = name[v.strip().upper()]
            if v != v.strip():
                return " " + core
            return core.lower() if v == v.lower() else core
        loans["branch_id"] = loans["branch_id"].map(to_name)
        loans = loans.rename(columns={"branch_id": "branch"})
    elif variant == "E":
        items.insert(2, "classification", items["item_type"] + " / " + items["category"])
        items = items.drop(columns=["item_type", "category"])

    files = {"branches": branches, "members": members, "items": items}
    if variant == "C":
        half = loans["loan_date"].map(parse_date).dt.month <= 6
        h1, h2 = loans[half], loans[~half]
        cols = list(h2.columns)
        cols[4], cols[6] = cols[6], cols[4]  # swap loan_date / return_date positions
        files["loans_2025_H1"] = h1
        files["loans_2025_H2"] = h2[cols]
    else:
        files["loans"] = loans
    for k, df in files.items():
        df.to_csv(f"{out}/{k}.csv", index=False)
    return list(files), dropped


def data_dictionary(variant, files):
    d = {
        "branches": "branch_id, branch_name, area, opened_year",
        "members": "member_id, join_date, age_band, membership_type (Junior / Standard / Concession), home_branch_id, postcode",
        "items": "item_id, title, item_type, category, replacement_cost (GBP), home_branch_id",
        "loans": "loan_id, member_id, item_id, branch_id (branch where the loan was made), loan_date, due_date, "
                 "return_date (blank = not returned), fine_charged (GBP)",
    }
    if variant == "E":
        d["items"] = "item_id, title, classification (item type / category), replacement_cost (GBP), home_branch_id"
    if variant == "B":
        d["loans"] = d["loans"].replace("fine_charged (GBP)", "fine_pence (whole pence)")
    if variant == "D":
        d["loans"] = d["loans"].replace("branch_id (branch where", "branch (name of the branch where")
    lines = ["# Data dictionary", "", "Fictional data for a community library service, calendar year 2025.", ""]
    for f in files:
        key = "loans" if f.startswith("loans") else f
        lines.append(f"- **{f}.csv**: {d[key]}")
    if variant == "C":
        lines.append("")
        lines.append("loans are split by loan date: H1 = January to June, H2 = July to December. "
                     "Check the column order in each file.")
    lines += ["", "The files are exactly as exported from the library's systems. "
                  "Checking them before you rely on them is part of the task.", ""]
    return "\n".join(lines)


# ---------------------------------------------------------------- marker list
def write_marker(path, sid, seed, variant, log, dropped, ref, by_branch, by_month):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# Marker's list: library data for student ID {sid} (PRIVATE)\n\n")
        f.write(f"Seed (from ID): {seed}. Layout variant: **{variant}**: {VARIANT_TEXT[variant]}\n\n")
        f.write("The issues are ordinary, not traps. Use this to judge how thorough a student's checking was, "
                "not as a checklist every student must complete.\n\n")
        f.write("## Injected issues\n\n")
        f.write("Counts are for the original rows. Duplicate rows were appended afterwards, so a few affected rows appear twice.\n\n")
        f.write("| Issue | Rows affected |\n| --- | --- |\n")
        for k, v in log.items():
            if k in dropped:
                f.write(f"| {k.replace('_', ' ')} | not applicable in variant {variant} (converted to pence) |\n")
            else:
                f.write(f"| {k.replace('_', ' ')} | {v} |\n")
        f.write("\n## Issues arising from the design of the data (no injection)\n\n")
        f.write("- Return date blank: either a loan still out (recent loans) or an item never returned (old loans). The data does not say which.\n")
        f.write("- Zero fines on all Children items (a fine-free category), plus blank fines on some loans.\n")
        f.write("- Slash dates (dd/mm/yyyy) are ambiguous if read as mm/dd/yyyy; the format is consistent within a row.\n")
        f.write("- Loan periods differ by item type (Book 21, Audiobook 14, DVD 7 days), so lateness depends on joining to items.\n\n")
        f.write("## Variant-specific step\n\n")
        f.write(f"- {VARIANT_TEXT[variant]}\n\n")
        f.write("## Reference numbers after one defensible cleaning approach\n\n")
        f.write("Approach: drop exact duplicate loan rows, trim and upper-case branch_id, parse dates by format. "
                "Figures are the same whichever layout the student received. "
                "Other defensible choices (for example how to treat orphan keys) will change some figures.\n\n")
        for k, v in ref.items():
            f.write(f"- {k.replace('_', ' ')}: {v}\n")
        f.write("\n### Loans per branch (after cleaning)\n\n| branch_id | loans |\n| --- | --- |\n")
        for b, v in by_branch.items():
            f.write(f"| {b} | {v} |\n")
        f.write("\n### Loans per month (after cleaning)\n\n| month | loans |\n| --- | --- |\n")
        for m_, v in by_month.items():
            f.write(f"| {m_} | {v} |\n")


# ------------------------------------------------------------------- public
def generate(student_id, out_dir=None, marker=False, marker_dir="marker_lists", variant=None, quiet=False):
    """Create this student's dataset. `variant` overrides the ID-derived layout (testing only)."""
    sid = clean_id(student_id)
    seed = seed_from_id(sid)
    variant = variant or variant_from_id(sid)
    out_dir = out_dir or f"library_data_{re.sub(r'[^A-Za-z0-9]', '', sid)}"
    with tempfile.TemporaryDirectory() as tmp:
        log = build_baseline(seed, tmp)
        files, dropped = apply_variant(variant, tmp, out_dir)
        with open(f"{out_dir}/data_dictionary.md", "w", encoding="utf-8") as f:
            f.write(data_dictionary(variant, files))
        if marker:
            ref, by_branch, by_month = reference_numbers(tmp)
            os.makedirs(marker_dir, exist_ok=True)
            write_marker(f"{marker_dir}/marker_{re.sub(r'[^A-Za-z0-9]', '', sid)}.md",
                         sid, seed, variant, log, dropped, ref, by_branch, by_month)
    if not quiet:
        print(f"Created {out_dir}/ with: " + ", ".join(f"{f}.csv" for f in files) + ", data_dictionary.md")
    return out_dir, variant


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("student_id", nargs="?")
    ap.add_argument("--ids", help="text file with one student ID per line (marker use)")
    ap.add_argument("--marker", action="store_true", help="also write the private marker list")
    ap.add_argument("--variant", choices=list("ABCDE"), help="override layout (testing only)")
    a = ap.parse_args()
    if a.ids:
        rows = []
        for line in open(a.ids, encoding="utf-8"):
            sid = line.strip()
            if sid:
                _, v = generate(sid, out_dir=f"all_students/library_data_{re.sub(r'[^A-Za-z0-9]', '', sid)}",
                                marker=a.marker, quiet=True)
                rows.append((sid, v))
        os.makedirs("marker_lists", exist_ok=True)
        pd.DataFrame(rows, columns=["student_id", "variant"]).to_csv("marker_lists/variant_summary.csv", index=False)
        print(f"Generated {len(rows)} datasets in all_students/")
    else:
        sid = a.student_id or input("Enter your student ID: ")
        generate(sid, marker=a.marker, variant=a.variant)


if __name__ == "__main__":
    main()

Overwriting generate_library_data_by_id.py


## Step 2: Create your data

Run the cell and type your student ID when asked. Use the ID exactly as it appears on your student card. A different ID gives different data, so check you typed it correctly.

In [ ]:
from generate_library_data_by_id import generate

student_id = "5782076"
data_dir, _ = generate(student_id)

Created library_data_5782076/ with: branches.csv, members.csv, items.csv, loans_2025_H1.csv, loans_2025_H2.csv, data_dictionary.md


## Step 3: Read your data dictionary

This describes the files you have been given. Read it before you open the data.

In [ ]:
print(open(f"{data_dir}/data_dictionary.md", encoding="utf-8").read())

# Data dictionary

Fictional data for a community library service, calendar year 2025.

- **branches.csv**: branch_id, branch_name, area, opened_year
- **members.csv**: member_id, join_date, age_band, membership_type (Junior / Standard / Concession), home_branch_id, postcode
- **items.csv**: item_id, title, item_type, category, replacement_cost (GBP), home_branch_id
- **loans_2025_H1.csv**: loan_id, member_id, item_id, branch_id (branch where the loan was made), loan_date, due_date, return_date (blank = not returned), fine_charged (GBP)
- **loans_2025_H2.csv**: loan_id, member_id, item_id, branch_id (branch where the loan was made), loan_date, due_date, return_date (blank = not returned), fine_charged (GBP)

loans are split by loan date: H1 = January to June, H2 = July to December. Check the column order in each file.

The files are exactly as exported from the library's systems. Checking them before you rely on them is part of the task.



## Step 4a: Load the files (optional - if you are working in Colab)

This loads every CSV file in your folder into a dictionary called `tables`, exactly as it is in the file, with no cleaning. Your file list may differ from a classmate's.

In [ ]:
import glob
import os

import pandas as pd
from IPython.display import display

tables = {}
for path in sorted(glob.glob(f"{data_dir}/*.csv")):
    name = os.path.splitext(os.path.basename(path))[0]
    tables[name] = pd.read_csv(path)

for name, df in tables.items():
    print(f"\n=== {name}: {df.shape[0]:,} rows, {df.shape[1]} columns ===")
    display(df.head(3))


=== branches: 8 rows, 4 columns ===


,branch_id,branch_name,area,opened_year
0,B01,Abbeygate,North,1962
1,B02,Brookfield,North,1974
2,B03,Castlemere,East,1988



=== items: 1,200 rows, 6 columns ===


,item_id,title,item_type,category,replacement_cost,home_branch_id
0,I0001,The Little Stranger,DVD,Film,11.63,B07
1,I0002,The Little Forest,Book,Children,13.72,B02
2,I0003,The Winter Engine,Book,non-fiction,23.09,B04



=== loans_2025_H1: 2,824 rows, 8 columns ===


,loan_id,member_id,item_id,branch_id,loan_date,due_date,return_date,fine_charged
0,L000001,M00372,I0462,B02,2025-01-01,2025-01-22,2025-01-19,NaN
1,L000002,M00388,I0896,B01,2025-01-01,2025-01-22,2025-01-11,NaN
2,L000003,M00345,I0958,B08,01/01/2025,22/01/2025,15/01/2025,0.00



=== loans_2025_H2: 3,236 rows, 8 columns ===


,loan_id,member_id,item_id,branch_id,return_date,due_date,loan_date,fine_charged
0,L002800,M00012,I0566,B03,2025-07-04,2025-07-08,2025-07-01,NaN
1,L002801,M00118,I0842,B06,2025-07-02,2025-07-22,2025-07-01,0.00
2,L002802,M00465,I0921,b07,06 Jul 2025,22 Jul 2025,01 Jul 2025,0.00



=== members: 832 rows, 6 columns ===


,member_id,join_date,age_band,membership_type,home_branch_id,postcode
0,M00001,29/01/2017,60+,Standard,B02,ZZ6 7RZ
1,M00002,2020-04-16,35-59,Standard,B04,ZZ5 7DZ
2,M00003,2023-01-16,18-34,Standard,B03,ZZ7 1ZR


# IB9SR0 Analytics Engineering – Library data pipeline
**Student ID:** 5782076

This notebook runs the whole pipeline from start to finish.

Use **Runtime → Run all**.

| Part | What it does |
|---|---|
| Part 0 – Data generation | Course code (unchanged apart from the student ID): creates the raw CSV files |
| Part 1 – Pipeline settings | File paths, table names and expected columns, kept separate from the logic |
| Part 2 – Bronze layer | Loads the raw files |
| Part 3 – Silver layer | Cleans the raw data into neat tables |
| Part 4 – Gold layer | Creates business-ready tables for Tableau |

---
# Part 1 – Pipeline settings

This part describes the overall settings for the pipeline: where the data files are stored, the name of the database, and the columns expected in each table.

These are kept separate from the code in the later parts, so no file names or table names are hard-coded into the functions. This is because if this pipeline was to be reused next month with new files, only this cell would need to change and the rest of the pipeline would still work.

In [ ]:
!pip install -q duckdb
import duckdb, os
from datetime import datetime, timezone

# ---- PIPELINE SETTINGS (change these, not the functions below) ----
DATA_DIR = data_dir            # folder made by the generator in Step 2
DB_PATH  = "library.duckdb"    # DuckDB database file
BRONZE_SCHEMA = "bronze"

SOURCE_FILES = {
    "branches": ["branches.csv"],
    "members":  ["members.csv"],
    "items":    ["items.csv"],
    "loans":    ["loans_2025_H1.csv", "loans_2025_H2.csv"],
}

EXPECTED_COLUMNS = {
    "branches": ["branch_id", "branch_name", "area", "opened_year"],
    "members":  ["member_id", "join_date", "age_band", "membership_type", "home_branch_id", "postcode"],
    "items":    ["item_id", "title", "item_type", "category", "replacement_cost", "home_branch_id"],
    "loans":    ["loan_id", "member_id", "item_id", "branch_id", "loan_date", "due_date", "return_date", "fine_charged"],
}

In [ ]:
print(DATA_DIR)
print(SOURCE_FILES["loans"])

library_data_5782076
['loans_2025_H1.csv', 'loans_2025_H2.csv']


---
# Part 2 – Bronze layer: loading raw data

This part loads the raw files into DuckDB exactly as they were provided, so no cleaning is done here.

- All columns are loaded as text, so values such as `£19.94`, `b07` or `06 Jul 2025` stay exactly as they were.
- The two loans files are combined by column name, because H2 has its columns in a different order to H1. This is done in bronze as no values are changed, and `_source_file` keeps track of which file each row came from.
- Two load information columns are added: `_source_file` and `_loaded_at` (when it was loaded).
- The tables are replaced each time this part runs, so running it twice gives the same result instead of doubling the rows.

### 2.1 Load function

In [ ]:
def load_bronze(con, data_dir, source_files, schema="bronze"):
    """Load each source file as-is (all columns as text) and add load information.
    CREATE OR REPLACE means running it twice gives the same result."""
    con.execute(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    loaded_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    for table, files in source_files.items():
        paths = [os.path.join(data_dir, f) for f in files]
        con.execute(f"""
            CREATE OR REPLACE TABLE {schema}.{table} AS
            SELECT * EXCLUDE (filename),
                   regexp_extract(filename, '[^/]+$') AS _source_file,
                   TIMESTAMP '{loaded_at}'           AS _loaded_at
            FROM read_csv($paths, all_varchar = true, union_by_name = true,
                          filename = true, header = true)
        """, {"paths": paths})
        n = con.execute(f"SELECT COUNT(*) FROM {schema}.{table}").fetchone()[0]
        n_cols = len(con.execute(f"DESCRIBE {schema}.{table}").fetchall())
        print(f"{schema}.{table:9} {n:>6,} rows  {n_cols} columns  from {', '.join(files)}")


# ---- Run the bronze load ----
con = duckdb.connect(DB_PATH)
load_bronze(con, DATA_DIR, SOURCE_FILES, BRONZE_SCHEMA)

bronze.branches       8 rows  6 columns  from branches.csv
bronze.members      832 rows  8 columns  from members.csv
bronze.items      1,200 rows  8 columns  from items.csv
bronze.loans      6,060 rows  10 columns  from loans_2025_H1.csv, loans_2025_H2.csv


Two load information columns are added to every bronze table:
- `_source_file` shows which file each row came from. This is needed for loans, as H1 and H2 are combined into one table.
- `_loaded_at` shows when the data was loaded, so each run of the pipeline can be identified.

### 2.2 Bronze checks

Checking the bronze layer automatically every time it is loaded:
1. Row count: the rows in each bronze table match the rows in its source file(s), so nothing has been lost or added.
2. Columns: every column in `EXPECTED_COLUMNS` is present, including the two new columns within each table.



In [ ]:
def check_bronze(con, data_dir, source_files, expected_columns, schema="bronze"):
    """Check 1: bronze row count = rows in the source files (nothing lost or added).
    Check 2: every expected column is present."""
    all_ok = True
    for table, files in source_files.items():
        file_rows = sum(sum(1 for _ in open(os.path.join(data_dir, f), encoding="utf-8")) - 1
                        for f in files)
        n = con.execute(f"SELECT COUNT(*) FROM {schema}.{table}").fetchone()[0]
        cols = [r[0] for r in con.execute(f"DESCRIBE {schema}.{table}").fetchall()]
        missing = set(expected_columns[table]) - set(cols)
        ok = (n == file_rows) and not missing
        all_ok &= ok
        print(f"{'PASS' if ok else 'FAIL'}  {table:9} file rows={file_rows:>6,}  "
              f"bronze rows={n:>6,}  missing cols={sorted(missing) or 'none'}")
    return all_ok


# ---- Run the check straight after the load ----
assert check_bronze(con, DATA_DIR, SOURCE_FILES, EXPECTED_COLUMNS, BRONZE_SCHEMA), "Bronze checks failed"
print("All bronze checks passed")

PASS  branches  file rows=     8  bronze rows=     8  missing cols=none
PASS  members   file rows=   832  bronze rows=   832  missing cols=none
PASS  items     file rows= 1,200  bronze rows= 1,200  missing cols=none
PASS  loans     file rows= 6,060  bronze rows= 6,060  missing cols=none
All bronze checks passed


### 2.3 Preview bronze tables

Printing the first 5 rows of each bronze table with all columns, to check the data loaded as expected and the two load information columns were added.

In [33]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)   # show every column, none hidden

for table in SOURCE_FILES:
    print(f"\n=== bronze.{table} ===")
    display(con.sql(f"SELECT * FROM bronze.{table} LIMIT 5").df())

# Check H2 rows line up with H1 after combining by column name
print("\n=== bronze.loans – rows from H2 ===")
display(con.sql("SELECT * FROM bronze.loans WHERE _source_file LIKE '%H2%' LIMIT 5").df())


=== bronze.branches ===


,branch_id,branch_name,area,opened_year,_source_file,_loaded_at
0,B01,Abbeygate,North,1962,branches.csv,2026-11-18 18:36:24
1,B02,Brookfield,North,1974,branches.csv,2026-11-18 18:36:24
2,B03,Castlemere,East,1988,branches.csv,2026-11-18 18:36:24
3,B04,Dunmore,East,1995,branches.csv,2026-11-18 18:36:24
4,B05,Elmstead,South,1971,branches.csv,2026-11-18 18:36:24



=== bronze.members ===


,member_id,join_date,age_band,membership_type,home_branch_id,postcode,_source_file,_loaded_at
0,M00001,29/01/2017,60+,Standard,B02,ZZ6 7RZ,members.csv,2026-11-18 18:36:24
1,M00002,2020-04-16,35-59,Standard,B04,ZZ5 7DZ,members.csv,2026-11-18 18:36:24
2,M00003,2023-01-16,18-34,Standard,B03,ZZ7 1ZR,members.csv,2026-11-18 18:36:24
3,M00004,25/07/2016,18-34,Standard,B01,ZZ5 2YH,members.csv,2026-11-18 18:36:24
4,M00005,2022-10-24,60+,Concession,B08,ZZ5 8RB,members.csv,2026-11-18 18:36:24



=== bronze.items ===


,item_id,title,item_type,category,replacement_cost,home_branch_id,_source_file,_loaded_at
0,I0001,The Little Stranger,DVD,Film,11.63,B07,items.csv,2026-11-18 18:36:24
1,I0002,The Little Forest,Book,Children,13.72,B02,items.csv,2026-11-18 18:36:24
2,I0003,The Winter Engine,Book,non-fiction,23.09,B04,items.csv,2026-11-18 18:36:24
3,I0004,The Secret Orchard,Book,Non-fiction,8.76,B07,items.csv,2026-11-18 18:36:24
4,I0005,The Winter Stranger,DVD,Children,12.9,B04,items.csv,2026-11-18 18:36:24



=== bronze.loans ===


,loan_id,member_id,item_id,branch_id,loan_date,due_date,return_date,fine_charged,_source_file,_loaded_at
0,L000001,M00372,I0462,B02,2025-01-01,2025-01-22,2025-01-19,None,loans_2025_H1.csv,2026-11-18 18:36:24
1,L000002,M00388,I0896,B01,2025-01-01,2025-01-22,2025-01-11,None,loans_2025_H1.csv,2026-11-18 18:36:24
2,L000003,M00345,I0958,B08,01/01/2025,22/01/2025,15/01/2025,0.00,loans_2025_H1.csv,2026-11-18 18:36:24
3,L000004,M00383,I0075,B04,2025-01-01,2025-01-08,2025-01-04,0.00,loans_2025_H1.csv,2026-11-18 18:36:24
4,L000005,M00740,I0174,B06,2025-01-01,2025-01-22,2025-01-30,1.60,loans_2025_H1.csv,2026-11-18 18:36:24



=== bronze.loans – rows from H2 ===


,loan_id,member_id,item_id,branch_id,loan_date,due_date,return_date,fine_charged,_source_file,_loaded_at
0,L002800,M00012,I0566,B03,2025-07-01,2025-07-08,2025-07-04,None,loans_2025_H2.csv,2026-11-18 18:36:24
1,L002801,M00118,I0842,B06,2025-07-01,2025-07-22,2025-07-02,0.00,loans_2025_H2.csv,2026-11-18 18:36:24
2,L002802,M00465,I0921,b07,01 Jul 2025,22 Jul 2025,06 Jul 2025,0.00,loans_2025_H2.csv,2026-11-18 18:36:24
3,L002803,M00690,I0930,B01,2025-07-01,2025-07-15,2025-07-04,0.00,loans_2025_H2.csv,2026-11-18 18:36:24
4,L002804,M00202,I0472,B08,2025-07-01,2025-07-22,2025-07-09,None,loans_2025_H2.csv,2026-11-18 18:36:24
